# Role Generation for Spider Database Tables

This notebook performs role-based access control (RBAC) analysis for the Spider database collection using LLM.

### 1. Setup LLM Oracle instance

In [1]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path('/home/feiy/Role-SQL-benchmark')
sys.path.append(str(project_root))

# Import required modules
from src.role_parser import RoleGenerator, ParallelRoleGenerator
from dotenv import load_dotenv
import os
import json
from datetime import datetime
import logging
import random
import importlib
import src.llm_oracle as oracle

# Setup output directories
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_dir = project_root / 'logs'
output_dir = project_root / 'outputs'

for directory in [log_dir, output_dir]:
    directory.mkdir(exist_ok=True)

# Configure logging
log_file = log_dir / f'role_assignment_{timestamp}.log'
logging.getLogger().handlers.clear()

logger = logging.getLogger('role_assignment')
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(str(log_file))
console_handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

for handler in [file_handler, console_handler]:
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.propagate = False
logger.info(f"Starting new session at {timestamp}")
logger.info(f"Log file: {log_file}")
logger.info(f"Output directory: {output_dir}")

2025-09-16 21:46:23,311 - INFO - Starting new session at 20250916_214623
2025-09-16 21:46:23,312 - INFO - Log file: /home/feiy/Role-SQL-benchmark/logs/role_assignment_20250916_214623.log
2025-09-16 21:46:23,313 - INFO - Output directory: /home/feiy/Role-SQL-benchmark/outputs
2025-09-16 21:46:23,312 - INFO - Log file: /home/feiy/Role-SQL-benchmark/logs/role_assignment_20250916_214623.log
2025-09-16 21:46:23,313 - INFO - Output directory: /home/feiy/Role-SQL-benchmark/outputs


/home/feiy/anaconda3/envs/llm4db/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load environment variables and API keys
load_dotenv()
importlib.reload(oracle)

<module 'src.llm_oracle' from '/home/feiy/Role-SQL-benchmark/src/llm_oracle/__init__.py'>

In [3]:
# DeepSeek demo
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not DEEPSEEK_API_KEY:
    print("Warning: Cannot find DEEPSEEK_API_KEY in environment")
    print("Please set it in the .env file or environment")
if not OPENAI_API_KEY:
    print("Warning: Cannot find OPENAI_API_KEY in environment")
    print("Please set it in the .env file or environment")


In [4]:
def create_oracle(model_name, api_key):
    return oracle.Oracle(model_name, api_key)

#### 1.1. Deepseek test

##### 1.1(a) deepseek demo test

In [5]:
# if not DEEPSEEK_API_KEY:
#     print("Warning: DEEPSEEK_API_KEY not found in environment variables")
#     print("Please set it in your .env file or environment")
# else:
#     # Create Oracle instance with DeepSeek model
#     oracle = Oracle(model="deepseek-chat", apikey=DEEPSEEK_API_KEY)
    
#     # Test the model
#     test_response = oracle.query(
#         prompt_sys="You are a helpful assistant.",
#         prompt_user="Say Hi.",
#         temp=0.7,
#         top_p=0.9
#     )
    
#     print("Model Response:")
#     print("="*50)
#     print(test_response['answer'])

##### 1.1(b) prompt caching test

In [6]:
# # Create two identical requests to test caching
# prompt_sys = """You are a helpful assistant. You should:
# 1. Be concise and clear in your responses
# 2. Always strive to provide accurate information
# 3. Maintain a professional and friendly tone
# 4. Use appropriate formatting when needed
# 5. Ask for clarification if something is unclear"""

# prompt_user = """Please introduce yourself and tell me about your capabilities.
# Make sure to mention:
# 1. Your name
# 2. Your main areas of expertise
# 3. How you can help users
# 4. Any limitations users should be aware of"""

In [7]:
# def make_query(prompt_sys, prompt_user):
#     """Execute query and return response"""
#     response = oracle.query(
#         prompt_sys=prompt_sys,
#         prompt_user=prompt_user,
#     )
#     # if response.get('answer'):
#     #     print(f"Response: {response['answer']}")
#     return response

# def print_cache_stats(response, label=None):
#     """Print cache statistics for a response"""
#     if label:
#         print(f"\n=== {label} Statistics ===")
    
#     usage = response.get('usage', {})
#     prompt_details = usage.get('prompt_tokens_details', None)
    
#     print("\nToken Statistics:")
#     print(f"  Prompt tokens: {usage.get('prompt_tokens', 0)}")
#     print(f"  Completion tokens: {usage.get('completion_tokens', 0)}")
#     print(f"  Total tokens: {usage.get('total_tokens', 0)}")
    
#     print("\nCache Statistics:")
#     if prompt_details:
#         if isinstance(prompt_details, str):
#             print(f"  Raw info: {prompt_details}")
#         else:
#             cached = getattr(prompt_details, 'cached_tokens', 0)
#             non_cached = usage.get('prompt_tokens', 0) - cached
#             print(f"  Cached tokens: {cached}")
#             print(f"  Non-cached tokens: {non_cached}")
#             if cached > 0:
#                 print(f"  Cache hit rate: {(cached / usage.get('prompt_tokens', 1)) * 100:.1f}%")

In [8]:
# response1 = make_query(prompt_sys, prompt_user)
# response2 = make_query(prompt_sys, prompt_user)
# response3 = make_query(prompt_sys, prompt_user)

# print_cache_stats(response1, "First Call")
# print_cache_stats(response2, "Second Call")
# print_cache_stats(response3, "Third Call")

#### 1.2. OpenAI Test

In [9]:
# Example: create Oracle instance (model and api_key should be set according to your environment)

# MODEL_NAME = 'gpt-4o'  # or any supported model
# create_oracle_instance = create_oracle(model_name=MODEL_NAME, api_key=OPENAI_API_KEY)
# print(f"Oracle instance created for model: {MODEL_NAME}")

In [10]:
# build a simple prompt for testing

# prompt_sys = """You are a helpful assistant. You should:
# 1. Be concise and clear in your responses
# 2. Always strive to provide accurate information
# 3. Maintain a professional and friendly tone
# 4. Use appropriate formatting when needed
# 5. Ask for clarification if something is unclear"""
# prompt_user = """Please introduce yourself and tell me about your capabilities.
# Make sure to mention:
# 1. Your name
# 2. Your main areas of expertise
# 3. How you can help users
# 4. Any limitations users should be aware of"""
# response = openai_oracle_instance.query(
#     prompt_sys=prompt_sys,
#     prompt_user=prompt_user,
# )
# print(f"Response: {response['answer']}")

### 2. Role Assignment for Spider Database Tables

This section aims to:
1. Read schema information from Spider database
2. Use LLM to generate appropriate roles for each table
3. Test the role assignment with sample cases

In [11]:
# Setup Spider database path
SPIDER_ROOT = project_root / 'data/spider/database'

def get_db_folders():
    """Get list of database folders in Spider dataset"""
    if not SPIDER_ROOT.exists():
        logger.warning(f"Spider database directory not found at {SPIDER_ROOT}. Creating it now.")
        SPIDER_ROOT.mkdir(parents=True, exist_ok=True)
        return []
    
    return [d for d in SPIDER_ROOT.iterdir() if d.is_dir()]

# Get all database folders
db_folders = get_db_folders()
logger.info(f"Found {len(db_folders)} databases in Spider dataset")

2025-09-16 21:46:23,386 - INFO - Found 166 databases in Spider dataset


#### 2.1 Prompt Design for Role Assignment

The prompt is designed to:
1. Provide clear context about the task (Role-Based Access Control)
2. Guide the LLM to analyze table schema and relationships
3. Generate appropriate role names and descriptions
4. Consider security implications
5. Maintain consistency across different tables

In [12]:
# Process databases with role assignment
N_SAMPLES = 12
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
logger.info(f"Starting role assignment process for {N_SAMPLES} databases")

# Initialize parallel role generator
generator = ParallelRoleGenerator(model="deepseek-chat", api_key=DEEPSEEK_API_KEY, n_workers=10)
logger.info(f"Initialized ParallelRoleGenerator with model: deepseek-chat")

# Select random databases to process
test_dbs = random.sample(db_folders, N_SAMPLES)
db_names = [db.name for db in test_dbs]
logger.info(f"Selected databases: {db_names}")

# Process databases in parallel
logger.info("Starting parallel processing of databases")
# Include database paths for table validation
sqlite_paths = {db.name: str(db / f"{db.name}.sqlite") for db in test_dbs}
results = generator.process_databases_parallel(test_dbs, sqlite_paths=sqlite_paths)

# Store results
role_assignments = {}
processed_count = 0
total_roles = 0

for result in results:
    if result and result.get('roles'):
        processed_count += 1
        roles_count = len(result['roles'])
        total_roles += roles_count
        role_assignments[result['database']] = result['roles']
    else:
        logger.error(f"Failed to process one of the databases")

# Prepare metadata
assignments_data = {
    'assignments': role_assignments,
    'metadata': {
        'timestamp': timestamp,
        'total_databases': len(test_dbs),
        'processed_databases': processed_count,
        'total_roles_generated': total_roles
    }
}

# Save results if we have any
if role_assignments:
    output_file = generator.save_assignments_parallel(assignments_data, output_dir, timestamp)

logger.info(f"\nProcess completed:")
logger.info(f"- Databases processed: {processed_count}/{len(test_dbs)}")
logger.info(f"- Total roles generated: {total_roles}")

2025-09-16 21:46:23,407 - INFO - Starting role assignment process for 12 databases
2025-09-16 21:46:23,428 - INFO - Initialized ParallelRoleGenerator with model: deepseek-chat
2025-09-16 21:46:23,428 - INFO - Selected databases: ['music_2', 'customer_deliveries', 'roller_coaster', 'farm', 'culture_company', 'department_management', 'pets_1', 'student_transcripts_tracking', 'insurance_fnol', 'hospital_1', 'cre_Drama_Workshop_Groups', 'allergy_1']
2025-09-16 21:46:23,429 - INFO - Starting parallel processing of databases
2025-09-16 21:46:23,430 - INFO - Processing 12 databases (0 skipped)
2025-09-16 21:46:23,430 - INFO - Starting parallel API calls with 10 workers
2025-09-16 21:46:23,428 - INFO - Initialized ParallelRoleGenerator with model: deepseek-chat
2025-09-16 21:46:23,428 - INFO - Selected databases: ['music_2', 'customer_deliveries', 'roller_coaster', 'farm', 'culture_company', 'department_management', 'pets_1', 'student_transcripts_tracking', 'insurance_fnol', 'hospital_1', 'cre

Processing Items: 100%|██████████| 12/12 [00:25<00:00,  2.12s/it]

2025-09-16 21:46:48,868 - INFO - Summary Statistics: Total API Calls: 12, Total Prompt Tokens: 57121, Total Completion Tokens: 1917, Total Cached Tokens: 14080, Overall Cache Hit Rate: 24.6%
2025-09-16 21:46:48,871 - INFO - Generated 4 roles for music_2 - Cached Tokens: 448; Uncached tokens: 480; Hit rates: 48.3%
2025-09-16 21:46:48,872 - INFO - Generated 5 roles for customer_deliveries - Cached Tokens: 448; Uncached tokens: 9698; Hit rates: 4.4%
2025-09-16 21:46:48,873 - INFO - Generated 3 roles for roller_coaster - Cached Tokens: 448; Uncached tokens: 531; Hit rates: 45.8%
2025-09-16 21:46:48,874 - INFO - Generated 4 roles for farm - Cached Tokens: 1728; Uncached tokens: 40; Hit rates: 97.7%
2025-09-16 21:46:48,875 - INFO - Generated 3 roles for culture_company - Cached Tokens: 448; Uncached tokens: 1236; Hit rates: 26.6%
2025-09-16 21:46:48,876 - INFO - Generated 4 roles for department_management - Cached Tokens: 448; Uncached tokens: 1038; Hit rates: 30.1%
2025-09-16 21:46:48,871 -